<div dir="rtl" style="text-align:right">

# שבוע 1 · פוטנציאל כימי כתלות בטמפרטורה

*הדגמה לתרגול 1 · מבוא למצב מוצק תשפ״ז · אוניברסיטת תל אביב*

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/yaminbens/SolidState/blob/main/week01.ipynb) &nbsp;&nbsp; [⬇ הורדת המחברת](https://github.com/yaminbens/SolidState/raw/main/week01.ipynb)

> 🧭 **איך עובדים עם המחברת?** ראשית, לפני כל גרף, נסו לנחש מה תראו. שנית, הריצו (*Run all*, או תא אחר תא); כל המחברת רצה בתוך דקה, ב-Colab או מקומית (`numpy`, `scipy`, `matplotlib`). לבסוף, ענו לעצמכם על שאלות הדיון שבסוף. אין צורך לכתוב קוד: הסעיף החישובי בתרגיל הבית נשען על ההדגמה הזו.

## תיאור ההדגמה
בטמפרטורה סופית האכלוס נקבע על ידי $f_{FD}$, והפוטנציאל הכימי $\mu$ נקבע מהדרישה שמספר החלקיקים נשאר קבוע:
$$n=\int_0^\infty g(\varepsilon)\,f_{FD}(\varepsilon;\mu,T)\,d\varepsilon .$$
זו משוואה לא-ליניארית עבור $\mu(T)$, והמחברת פותרת אותה מספרית (מציאת שורש) בכל טמפרטורה.

**יחידות:** $\varepsilon_F=1$, $k_B=1$ (כלומר $T$ נמדד ב-$T_F$) ו-$n=1$.

</div>

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scipy.integrate import quad
from scipy.optimize import brentq
from scipy.special import expit   # expit(x) = 1/(1+exp(-x)), numerically stable

<div dir="rtl" style="text-align:right">

### ⚙️ פרמטרים
- `d` – המימד של הגז. צפיפות המצבים היא $g(\varepsilon)=c\,\varepsilon^{d/2-1}$, כאשר הקבוע $c$ נקבע כך שב-$T=0$ מתקבל $n=\int_0^{\varepsilon_F}g\,d\varepsilon=1$ (כלומר $c=d/2$).
- `T_max` – הטמפרטורה הגבוהה ביותר בגרפים, ביחידות של $T_F$.

</div>

In [ ]:
# ---- Parameters ----
d = 2          # dimension of the gas
T_max = 2.0    # highest temperature shown (units of T_F)

<div dir="rtl" style="text-align:right">

### 1. התפלגות פרמי–דיראק וצפיפות המצבים

</div>

In [ ]:
def f_FD(e, mu, T):
    # Fermi-Dirac occupation, written with expit to avoid overflow
    return expit(-(e - mu) / T)

def g(e, d):
    # density of states ~ e^(d/2 - 1); the constant d/2 makes n = 1 at T = 0 (eps_F = 1)
    return (d / 2) * e ** (d / 2 - 1)

<div dir="rtl" style="text-align:right">

### 2. פתרון מספרי של $\mu(T)$
בכל טמפרטורה מחפשים את ה-$\mu$ שעבורו האינטגרל שווה ל-$n=1$. האינטגרל עולה עם $\mu$, ולכן מציאת השורש אמינה.

</div>

In [ ]:
def n_of_mu(mu, T, d):
    return quad(lambda e: g(e, d) * f_FD(e, mu, T), 0, np.inf, limit=200)[0]

def solve_mu(T, d, n=1.0):
    # root of n(mu) - n = 0; n(mu) is increasing in mu, so brentq is safe
    return brentq(lambda m: n_of_mu(m, T, d) - n, -60 * max(T, 0.02), 3.0, xtol=1e-12)

def mu_curve(Ts, d):
    return np.array([solve_mu(T, d) for T in Ts])

<div dir="rtl" style="text-align:right">

### 3. ההתפלגות בשלוש טמפרטורות
שימו לב: ב-$T\ll T_F$ ההתפלגות כמעט מדרגה, ו-$\mu$ קרוב ל-$\varepsilon_F$.

</div>

In [ ]:
x = np.linspace(0, 2, 600)
for T, ls in [(0.01, "-"), (0.1, "--"), (0.5, ":")]:
    mu = solve_mu(T, d)
    plt.plot(x, f_FD(x, mu, T), ls, label=f"T/T_F = {T},  mu = {mu:.4f}")
plt.axvline(1, color="0.7", lw=0.8)
plt.xlabel("epsilon / epsilon_F"); plt.ylabel("f_FD"); plt.title(f"d = {d}")
plt.legend(frameon=False); plt.show()

<div dir="rtl" style="text-align:right">

### 4. הגרף של $\mu(T)$
> 🔮 **נחשו:** כשמעלים את הטמפרטורה, האם $\mu$ עולה, יורד או נשאר קבוע? נמקו במשפט אחד לפני שמסתכלים על הגרף.

</div>

In [ ]:
Ts = np.linspace(0.02, T_max, 100)
plt.plot(Ts, mu_curve(Ts, d), label=f"d = {d}")
plt.axhline(1, color="0.7", lw=0.8); plt.axhline(0, color="0.7", lw=0.8)
plt.xlabel("T / T_F"); plt.ylabel("mu / epsilon_F"); plt.legend(frameon=False)
plt.show()

<div dir="rtl" style="text-align:right">

### 💬 שאלות לדיון
1. מהי ההתנהגות של $\mu$ ב-$T\ll T_F$? האם השינוי ביחס ל-$\varepsilon_F$ גדול או קטן?
2. מה קורה ל-$\mu$ כש-$T\gg T_F$? מה הסימן שלו, ולמה? (רמז: הגבול הקלאסי.)
3. מה מייחד את צפיפות המצבים בסביבת $\varepsilon_F$, ואיך זה קשור להתנהגות שראיתם?

> ✅ **סיימתם?** תשובות לשאלות הדיון אינן להגשה. הסעיף החישובי בתרגיל הבית נשען על המחברת הזו. בהצלחה!

</div>